In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

In [ ]:
matrix = pd.read_pickle('../data/preprocessed/user_item_matrix_baseline.pkl')
movies = pd.read_csv('../data/preprocessed/movies_cleaned.csv')

item_sim = cosine_similarity(matrix)
item_sim_df = pd.DataFrame(item_sim, index=matrix.index, columns=matrix.index)

In [ ]:
def get_recommendations(movie_title, top_n=10):
    """
    Finds and returns movies similar to the input title based on user ratings.
    """
    # Find the movieId for the given title (case-insensitive search)
    try:
        movie_row = movies[movies['title'].str.contains(movie_title, case=False, na=False)].iloc[0]
        movie_id = movie_row['movieId']
        actual_title = movie_row['title']
    except IndexError:
        return f"Movie '{movie_title}' not found in the database."

    # Get similarity scores for this movie, sorted in descending order
    # The first result will always be the movie itself (similarity = 1.0)
    similar_scores = item_sim_df[movie_id].sort_values(ascending=False)
    
    # Remove the input movie from the results
    similar_scores = similar_scores.drop(movie_id)
    
    # Take the top N results
    top_movie_ids = similar_scores.head(top_n).index
    
    # Build the final recommendation table
    recommendations = movies[movies['movieId'].isin(top_movie_ids)].copy()
    recommendations['similarity'] = recommendations['movieId'].map(similar_scores)
    
    print(f"Recommendations for: {actual_title}")
    return recommendations[['title', 'genres', 'similarity']].sort_values(by='similarity', ascending=False)

In [ ]:
get_recommendations("Forrest Gump", top_n=5)